In [ ]:
pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.17.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 13.1 which is incompatible.
dataproc-spark-connect 0.8.3 requires websockets>=14.0, but you have websockets 13.1 which is incompatible.


In [ ]:
import fastf1 as ff1
import pandas as pd
import numpy as np
from fastf1 import plotting
from sklearn.preprocessing import StandardScaler

In [ ]:
import os

# Ensure the cache directory exists
if not os.path.exists('cache'):
    os.makedirs('cache')

ff1.Cache.enable_cache('cache')

In [ ]:
# Configura el plotting de FastF1
plotting.setup_mpl()

# --- 1. Carga de Datos ---

session = ff1.get_session(2025, 'Italian Grand Prix', 'R')
session.load()
print(f"Sesión cargada: {session.event.EventName} {session.name}")

core           INFO 	Loading data for Italian Grand Prix - Race [v3.6.1]
INFO:fastf1.fastf1.core:Loading data for Italian Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

Sesión cargada: Italian Grand Prix Race


In [ ]:
# --- 2. Definición de Zonas de Interés (ZOI) ---
# Definimos zonas de análisis con sub-partes: 'braking', 'apex', 'exit'
circuit_info = session.get_circuit_info()
analysis_zones = {
    'T1_Rettifilo': {
        'braking': (circuit_info.corners.loc[0]['Distance'] - 350, circuit_info.corners.loc[0]['Distance']),
        'apex': (circuit_info.corners.loc[0]['Distance'] - 50, circuit_info.corners.loc[1]['Distance'] + 50),
        'exit': (circuit_info.corners.loc[1]['Distance'], circuit_info.corners.loc[1]['Distance'] + 250)
    },
    'T4_Roggia': {
        'braking': (circuit_info.corners.loc[3]['Distance'] - 300, circuit_info.corners.loc[3]['Distance']),
        'apex': (circuit_info.corners.loc[3]['Distance'] - 50, circuit_info.corners.loc[4]['Distance'] + 50),
        'exit': (circuit_info.corners.loc[4]['Distance'], circuit_info.corners.loc[4]['Distance'] + 250)
    },
    'T11_Alboreto': {
        'braking': (circuit_info.corners.loc[10]['Distance'] - 350, circuit_info.corners.loc[10]['Distance']),
        'apex': (circuit_info.corners.loc[10]['Distance'] - 100, circuit_info.corners.loc[10]['Distance'] + 100),
        'exit': (circuit_info.corners.loc[10]['Distance'] + 100, circuit_info.corners.loc[10]['Distance'] + 350)
    }
}
print(f"Definidas {len(analysis_zones)} zonas de análisis.")

Definidas 3 zonas de análisis.


In [ ]:
# --- 3. Funciones de Feature Engineering ---

def get_braking_point(telemetry_fragment):
    """Devuelve la distancia donde 'Brake' se activa."""
    try:
        braking_start_row = telemetry_fragment[telemetry_fragment['Brake'] > 0.1].first_valid_index()
        if braking_start_row is not None:
            return telemetry_fragment.loc[braking_start_row, 'Distance']
    except Exception as e:
        pass
    return np.nan

def get_min_apex_speed(telemetry_fragment):
    """Devuelve la velocidad mínima en el fragmento de vértice."""
    try:
        if not telemetry_fragment.empty:
            return telemetry_fragment['Speed'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_deceleration(telemetry_fragment):
    """Devuelve la desaceleración máxima (en m/s^2) mientras se frena."""
    try:
        # Filtra solo los momentos en que el freno está aplicado
        braking_fragment = telemetry_fragment[telemetry_fragment['Brake'] > 0.1]
        if not braking_fragment.empty:
            # Encuentra el valor mínimo de aceleración (máxima desaceleración)
            return braking_fragment['Acceleration'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_throttle_rate(telemetry_fragment):
    """Devuelve la tasa máxima de aplicación del acelerador (% por segundo)."""
    try:
        if not telemetry_fragment.empty:
            # Tasa de cambio del acelerador (0-100) por segundo
            throttle_rate = telemetry_fragment['Throttle'].diff() / telemetry_fragment['TimeDelta']

            # Encuentra la tasa positiva máxima
            max_rate = throttle_rate[throttle_rate > 0].max()

            if pd.notna(max_rate) and not np.isinf(max_rate):
                return max_rate
    except Exception as e:
        pass
    return np.nan


In [ ]:
# --- 4. Iteración por todos los pilotos y curvas (Multivariable) ---

driver_list = session.drivers
print(f"Analizando {len(driver_list)} pilotos...")

all_driver_features = []

for driver_code in driver_list:
    lap = session.laps.pick_drivers(driver_code).pick_fastest()

    if not isinstance(lap, pd.Series) or lap.empty:
        # print(f"Saltando a {driver_code} (sin vuelta rápida válida)")
        continue

    telemetry = lap.get_car_data().add_distance()

    # --- Pre-cálculo de Aceleración y TimeDelta (¡IMPORTANTE!) ---
    telemetry['Speed_ms'] = telemetry['Speed'] / 3.6  # Convertir km/h a m/s
    telemetry['TimeDelta'] = telemetry['Time'].diff().dt.total_seconds()
    telemetry['SpeedDelta'] = telemetry['Speed_ms'].diff()
    # Aceleración en m/s^2
    telemetry['Acceleration'] = telemetry['SpeedDelta'] / telemetry['TimeDelta']

    driver_data = {'Driver': driver_code}

    for zone_name, zone_parts in analysis_zones.items():

        # --- 1. Fragmentar la telemetría según las sub-zonas ---
        braking_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['braking'][0]) & (telemetry['Distance'] <= zone_parts['braking'][1])
        ]
        apex_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['apex'][0]) & (telemetry['Distance'] <= zone_parts['apex'][1])
        ]
        exit_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['exit'][0]) & (telemetry['Distance'] <= zone_parts['exit'][1])
        ]

        # --- 2. Calcular y guardar cada feature ---
        driver_data[f'{zone_name}_BrakingPoint'] = get_braking_point(braking_fragment)
        driver_data[f'{zone_name}_MaxDecel'] = get_max_deceleration(braking_fragment)
        driver_data[f'{zone_name}_ApexSpeed'] = get_min_apex_speed(apex_fragment)
        driver_data[f'{zone_name}_ThrottleRate'] = get_max_throttle_rate(exit_fragment)

    all_driver_features.append(driver_data)

Analizando 20 pilotos...


In [ ]:
# --- 5. Creación y Normalización del DataFrame ---

# Convertimos la lista de diccionarios en un DataFrame y eliminamos pilotos sin datos
df_raw_features = pd.DataFrame(all_driver_features).set_index('Driver').dropna()

display("\n--- DataFrame de Features (Datos Brutos) ---")
display(df_raw_features.head())

# Calculamos la media de cada feature (columna)
feature_averages = df_raw_features.mean()

# NORMALIZACIÓN: Restamos la media de la feature a cada piloto
df_normalized = df_raw_features - feature_averages

display("\n--- DataFrame Normalizado (Puntuación de Estilo) ---")
display(df_normalized.head())
display("\nInterpretación del DataFrame Normalizado:")
display("Valores positivos = 'más que el promedio' (ej. frenó más tarde, fue más rápido en vértice, etc.)")
display("Valores negativos = 'menos que el promedio'")


'\n--- DataFrame de Features (Datos Brutos) ---'

,T1_Rettifilo_BrakingPoint,T1_Rettifilo_MaxDecel,T1_Rettifilo_ApexSpeed,T1_Rettifilo_ThrottleRate,T4_Roggia_BrakingPoint,T4_Roggia_MaxDecel,T4_Roggia_ApexSpeed,T4_Roggia_ThrottleRate,T11_Alboreto_BrakingPoint,T11_Alboreto_MaxDecel,T11_Alboreto_ApexSpeed,T11_Alboreto_ThrottleRate
Driver,,,,,,,,,,,,
4,805.770000,-35.052910,73.0,62.500000,2008.020000,-88.541667,109.0,81.250000,4981.580278,-37.037037,207.0,6.25
81,768.865556,-48.611111,75.0,68.750000,2005.569167,-42.824074,113.0,115.000000,5016.669167,-34.722222,207.0,6.25
31,783.142222,-48.611111,68.0,68.322981,2003.894722,-40.895062,99.0,78.853047,5007.560278,-24.801587,193.0,5.00
14,774.203056,-41.666667,71.0,50.000000,2009.804444,-59.027778,100.0,112.500000,5021.503333,-29.166667,192.0,2.50


'\n--- DataFrame Normalizado (Puntuación de Estilo) ---'

,T1_Rettifilo_BrakingPoint,T1_Rettifilo_MaxDecel,T1_Rettifilo_ApexSpeed,T1_Rettifilo_ThrottleRate,T4_Roggia_BrakingPoint,T4_Roggia_MaxDecel,T4_Roggia_ApexSpeed,T4_Roggia_ThrottleRate,T11_Alboreto_BrakingPoint,T11_Alboreto_MaxDecel,T11_Alboreto_ApexSpeed,T11_Alboreto_ThrottleRate
Driver,,,,,,,,,,,,
4,22.774792,8.432540,1.25,0.106755,1.197917,-30.719522,3.75,-15.650762,-25.247986,-5.605159,7.25,1.25
81,-14.129653,-5.125661,3.25,6.356755,-1.252917,14.998071,7.75,18.099238,9.840903,-3.290344,7.25,1.25
31,0.147014,-5.125661,-3.75,5.929736,-2.927361,16.927083,-6.25,-18.047715,0.732014,6.630291,-6.75,0.00
14,-8.792153,1.818783,-0.75,-12.393245,2.982361,-1.205633,-5.25,15.599238,14.675069,2.265212,-7.75,-2.50


'\nInterpretación del DataFrame Normalizado:'

"Valores positivos = 'más que el promedio' (ej. frenó más tarde, fue más rápido en vértice, etc.)"

"Valores negativos = 'menos que el promedio'"

In [ ]:


# --- 6. ESTANDARIZACIÓN (Paso para Clustering) ---

print("\n--- Estandarizando datos para Clustering ---")

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_normalized)

# Creamos un nuevo DataFrame con los datos estandarizados
df_standardized = pd.DataFrame(
    data=scaled_data,
    index=df_normalized.index,
    columns=df_normalized.columns
)

print("\n--- DataFrame Estandarizado (Listo para Clustering/PCA/t-SNE) ---")
print(df_standardized.head())
print("\nInterpretación del DataFrame Estandarizado:")
print("Todas las features están ahora en la misma escala (Z-scores, media=0, std=1).")
print("Este es el input final para tu análisis de machine learning.")


--- Estandarizando datos para Clustering ---

--- DataFrame Estandarizado (Listo para Clustering/PCA/t-SNE) ---
        T1_Rettifilo_BrakingPoint  T1_Rettifilo_MaxDecel  \
Driver                                                     
4                        1.614805               1.496767   
81                      -1.001837              -0.909799   
31                       0.010424              -0.909799   
14                      -0.623392               0.322832   

        T1_Rettifilo_ApexSpeed  T1_Rettifilo_ThrottleRate  \
Driver                                                      
4                     0.483368                   0.014104   
81                    1.256757                   0.839813   
31                   -1.450105                   0.783398   
14                   -0.290021                  -1.637315   

        T4_Roggia_BrakingPoint  T4_Roggia_MaxDecel  T4_Roggia_ApexSpeed  \
Driver                                                                    
4        